# Imports and Data Setup


In [1]:
!git clone https://github.com/hallegirum/graff_extension.git
%cd graff_extension

Cloning into 'graff_extension'...
remote: Enumerating objects: 558, done.
remote: Counting objects: 100% (174/174), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 558 (delta 137), reused 60 (delta 25), pack-reused 384 (from 2)
Receiving objects: 100% (558/558), 53.96 MiB | 23.72 MiB/s, done.
Resolving deltas: 100% (226/226), done.
/content/graff_extension


In [ ]:
!pip install --no-cache-dir pyg-lib torch-scatter torch-sparse torch-cluster \
  -f https://data.pyg.org/whl/torch-2.10.0+cu128.html

!pip install --no-cache-dir torch-geometric torchdiffeq deeprobust

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch_scatter
import torch_sparse
import torch_geometric

In [ ]:
%cd src
!python run_GNN.py --dataset Roman-empire --w_style diag --lr 0.0021  --decay 0.0184  --dropout 0.30  --input_dropout 0.44  --hidden_dim 64 --time 2.0 --step_size 1 --geom_gcn_splits --num_splits 10

# FOSRA

In [3]:
"""
FoSR Code for comparison on synthetic datasets

"""


from torch_geometric.utils import to_networkx
from numba import jit, int64
import networkx as nx
import numpy as np
from math import inf


@jit(nopython=True)
def choose_edge_to_add(x, edge_index, degrees):
	# chooses edge (u, v) to add which minimizes y[u]*y[v]
	n = x.size
	m = edge_index.shape[1]
	y = x / ((degrees + 1) ** 0.5)
	products = np.outer(y, y)
	for i in range(m):
		u = edge_index[0, i]
		v = edge_index[1, i]
		products[u, v] = inf
	for i in range(n):
		products[i, i] = inf
	smallest_product = np.argmin(products)
	return (smallest_product % n, smallest_product // n)

@jit(nopython=True)
def compute_degrees(edge_index, num_nodes=None):
	# returns array of degrees of all nodes
	if num_nodes is None:
		num_nodes = np.max(edge_index) + 1
	degrees = np.zeros(num_nodes)
	m = edge_index.shape[1]
	for i in range(m):
		degrees[edge_index[0, i]] += 1
	return degrees

@jit(nopython=True)
def add_edge(edge_index, u, v):
	new_edge = np.array([[u, v],[v, u]])
	return np.concatenate((edge_index, new_edge), axis=1)

@jit(nopython=True)
def adj_matrix_multiply(edge_index, x):
	# given an edge_index, computes Ax, where A is the corresponding adjacency matrix
	n = x.size
	y = np.zeros(n)
	m = edge_index.shape[1]
	for i in range(m):
		u = edge_index[0, i]
		v = edge_index[1, i]
		y[u] += x[v]
	return y

@jit(nopython=True)
def compute_spectral_gap(edge_index, x):
	m = edge_index.shape[1]
	n = np.max(edge_index) + 1
	degrees = compute_degrees(edge_index, num_nodes=n)
	y = adj_matrix_multiply(edge_index, x / (degrees ** 0.5)) / (degrees ** 0.5)
	for i in range(n):
		if x[i] > 1e-9:
			return 1 - y[i]/x[i]
	return 0.

@jit(nopython=True)
def _edge_rewire(edge_index, edge_type, x=None, num_iterations=1, initial_power_iters=1):
	m = edge_index.shape[1]
	n = np.max(edge_index) + 1
	if x is None:
		x = 2 * np.random.random(n) - 1
	degrees = compute_degrees(edge_index, num_nodes=n)
	for i in range(initial_power_iters):
		x = x - x.dot(degrees ** 0.5) * (degrees ** 0.5)/sum(degrees)
		y = x + adj_matrix_multiply(edge_index, x / (degrees ** 0.5)) / (degrees ** 0.5)
		x = y / np.linalg.norm(y)
	for I in range(num_iterations):
		i, j = choose_edge_to_add(x, edge_index, degrees=degrees)
		edge_index = add_edge(edge_index, i, j)
		degrees[i] += 1
		degrees[j] += 1
		# edge_type = np.append(edge_type, 1)
		# edge_type = np.append(edge_type, 1)
		x = x - x.dot(degrees ** 0.5) * (degrees ** 0.5)/sum(degrees)
		y = x + adj_matrix_multiply(edge_index, x / (degrees ** 0.5)) / (degrees ** 0.5)
		x = y / np.linalg.norm(y)
	return edge_index, edge_type, x

def edge_rewire(edge_index, x=None, edge_type=None, num_iterations=50, initial_power_iters=5):
	m = edge_index.shape[1]
	n = np.max(edge_index) + 1
	if x is None:
		x = 2 * np.random.random(n) - 1
	if edge_type is None:
		edge_type = np.zeros(m, dtype=np.int64)
	return _edge_rewire(edge_index, edge_type=edge_type, x=x, num_iterations=num_iterations, initial_power_iters=initial_power_iters)

# GRAFF Extension

In [ ]:
## gradient-based rewiring
from itertools import product
import torch
from torch_geometric.utils import get_laplacian, remove_self_loops, add_remaining_self_loops, contains_self_loops, is_undirected
import torch_sparse
from collections import defaultdict
import numpy as np

def has_edge(edge_index, i, j):
  return bool(((edge_index[0]==i) & (edge_index[1]==j)).any())

def add_edge(edge_index, u, v):
  new_edge = torch.tensor([u,v], dtype=edge_index.dtype, device = edge_index.device)
  edge_index = torch.cat([edge_index, new_edge.unsqueeze(1)], dim=1)
  return edge_index

def get_neighbours(edge_index,i):
  neighbours = edge_index[1][edge_index[0]==i]
  return neighbours.tolist()

def build_neighbour_dict(edge_index):
    neigh = defaultdict(list)
    src = edge_index[0].tolist()
    dst = edge_index[1].tolist()
    for u, v in zip(src, dst):
        neigh[u].append(v)
    return neigh

def add_edge_to_neighbour_dict(neighbour_dict, u, v):
    new_dict = neighbour_dict.copy()
    new_dict[u] = neighbour_dict[u] + [v]
    new_dict[v] = neighbour_dict[v] + [u]
    return new_dict


def dirichlet_energy_grad(edge_index, n, X, edge_weight=None, norm_type=None):
  edge_index, L = get_laplacian(edge_index, edge_weight, norm_type)
  LX = torch_sparse.spmm(edge_index, L, n, n, X)
  return LX


def bottleneck_score(edge_index, n, X, eta=0.1, eps=1e-8):
    LX = dirichlet_energy_grad(edge_index, n, X)
    X_smooth = X - eta * LX

    LX_smooth = dirichlet_energy_grad(edge_index, n, X_smooth)
    diff = torch.norm(
        LX_smooth[edge_index[0]] - LX_smooth[edge_index[1]], dim=-1
    )**2
    mag = torch.norm(LX_smooth[edge_index[0]], dim=-1)**2 + \
          torch.norm(LX_smooth[edge_index[1]], dim=-1)**2
    return diff + mag



def bottleneck_score_v2(edge_index, n, X, eta=None, eps=1e-8):

    # Compute max degree for stable eta
    deg = torch.zeros(n, device=edge_index.device)
    deg.scatter_add_(
        0, edge_index[0],
        torch.ones(edge_index.shape[1], device=edge_index.device)
    )
    d_max = deg.max().item()

    # Stable eta: must satisfy eta < 1/d_max for unnormalised Laplacian
    if eta is None:
        eta = 0.9 / (d_max + 1e-8)

    LX = dirichlet_energy_grad(edge_index, n, X)
    X_smooth = X - eta * LX

    i = edge_index[0]
    j = edge_index[1]

    diff = torch.norm(X_smooth[i] - X_smooth[j], dim=-1)
    mag  = torch.norm(X_smooth[i], dim=-1) + torch.norm(X_smooth[j], dim=-1)

    score = diff / (mag + eps)
    return score

def evaluate_neighbourhood(X, edge_index, n, local_neigh, neigh_dict, eta=0.1):

    # Compute scores for ALL edges once — returns (m,) tensor
    all_scores = bottleneck_score_v2(edge_index, n, X)

    # Build edge-to-index lookup for fast access
    # Maps (u,v) -> index in edge_index
    edge_to_idx = {}
    for idx in range(edge_index.shape[1]):
        u = edge_index[0, idx].item()
        v = edge_index[1, idx].item()
        edge_to_idx[(u, v)] = idx

    local_score = 0.0
    count = 0
    seen = set()
    local_set = set(local_neigh)

    for u in local_neigh:
        for v in neigh_dict.get(u, []):
            if v not in local_set:
                continue
            a, b = min(u, v), max(u, v)
            if (a, b) in seen:
                continue
            seen.add((a, b))

            # Look up score for this specific edge
            if (u, v) in edge_to_idx:
                idx = edge_to_idx[(u, v)]
                local_score += all_scores[idx].item()
                count += 1

    return local_score / count if count > 0 else 0.0


def energy_gradient_rewire(edge_index, n, X, k,eta=0.1):

  """
  edge_index : (2, m) indicating that there is an edge between node i and node j
  n : number of nodes
  X : (n,d) features
  k : number of edges add
  adj : (n,n) adjacency matrix
  """
  for idx in range(k):
    edge_index, _ = remove_self_loops(edge_index)
    LX = dirichlet_energy_grad(edge_index,n,X)
    X_prop = X - eta * LX
    LX_prop = dirichlet_energy_grad(edge_index, n, X_prop)
    score = bottleneck_score(LX_prop[edge_index[0]],LX_prop[edge_index[1]])
    # print("min:", score.min().item())
    # print("max:", score.max().item())
    # print("std:", score.std().item())
    bottleneck= torch.argmax(score,dim=-1)
    node_i = edge_index[0,bottleneck].item()
    node_j = edge_index[1,bottleneck].item()
    print("node_i",node_i)
    print("node_j",node_j)
    neigh_i = get_neighbours(edge_index,node_i)
    neigh_j = get_neighbours(edge_index,node_j)
    # print("neighboursi", len(neigh_i))
    # print("neighboursj",len(neigh_j))

    max_score = (-float('inf'),(-1,-1))
    print("old_bn", score.min())
    local_neigh = list(set(neigh_i + neigh_j + [node_i, node_j]))
    old_score = evaluate_neighbourhood(X,edge_index,n,local_neigh)
    for (u,v) in product(neigh_i,neigh_j):

        if has_edge(edge_index,u,v) or u==v:
          continue
        rewired_edge_index = add_edge(edge_index,u,v)
        new_score = evaluate_neighbourhood(X,rewired_edge_index,n,local_neigh)
        relief =  old_score - new_score
        # print("old_score",old_score)
        # print("new_score",new_score)
        if max_score[0] < relief:
          max_score = (relief, (u,v))


    if max_score[1] != (-1,-1):
      print("edge_added")
      u,v = max_score[1]
      edge_index = add_edge(edge_index,u,v)
      edge_index = add_edge(edge_index,v,u)
      # LX_new = dirichlet_energy_grad(edge_index,n,X)
      # X = X - eta *LX_new

  edge_index, _ = add_remaining_self_loops(edge_index)
  print(edge_index.shape)
  return edge_index


def score_neighbourhood(LX, neighbour_dict, local_neigh):
    local_set = set(local_neigh)
    seen = set()
    total = 0.0
    count = 0

    for u in local_neigh:
        for v in neighbour_dict[u]:
            if v in local_set:
                a, b = (u, v) if u < v else (v, u)
                if (a, b) in seen:
                    continue
                seen.add((a, b))
                total += bottleneck_score(LX[u], LX[v]).item()
                count += 1

    return total / count if count > 0 else 0.0

def evaluate_neighbourhood_fast(LX_smooth, neighbour_dict, local_neigh, candidate_u, candidate_v):

    deg_u = len(neighbour_dict[candidate_u])
    deg_v = len(neighbour_dict[candidate_v])

    LX_approx = LX_smooth.clone()
    diff = LX_smooth[candidate_u] - LX_smooth[candidate_v]

    weight_u = 1.0 / (deg_u + 1)
    weight_v = 1.0 / (deg_v + 1)

    LX_approx[candidate_u] = LX_smooth[candidate_u] - weight_u * diff
    LX_approx[candidate_v] = LX_smooth[candidate_v] + weight_v * diff

    augmented_dict = add_edge_to_neighbour_dict(neighbour_dict, candidate_u, candidate_v)

    return score_neighbourhood(LX_approx, augmented_dict, local_neigh)

def energy_gradient_rewire_hybrid(edge_index, n, X, k, eta=0.1,
                                   recompute_every=1):
    """
    Hybrid: exact recomputation every recompute_every steps,
    approximate updates in between.

    recompute_every=1 recovers your exact dynamic method (slow)
    recompute_every=k recovers pure fast batch (fastest, least accurate)
    recompute_every=5 is a good middle ground
    """
    edge_index, _ = remove_self_loops(edge_index)
    edge_set = set(zip(edge_index[0].tolist(), edge_index[1].tolist()))
    neigh_dict = build_neighbour_dict(edge_index)

    # Initial exact computation
    LX = dirichlet_energy_grad(edge_index, n, X)
    X_smooth = X - eta * LX
    LX_smooth = dirichlet_energy_grad(edge_index, n, X_smooth)

    for idx in range(k):

        # Recompute exactly every recompute_every steps
        # This resets accumulated approximation error
        if idx > 0 and idx % recompute_every == 0:
            LX = dirichlet_energy_grad(edge_index, n, X)
            X_smooth = X - eta * LX
            LX_smooth = dirichlet_energy_grad(edge_index, n, X_smooth)

        # Score all edges using current LX_smooth (exact or approximate)
        # score = bottleneck_score(
        #     LX_smooth[edge_index[0]],
        #     LX_smooth[edge_index[1]]
        # )
        score= bottleneck_score_v2(edge_index,n,X)
        # score= tango_bottleneck_score(edge_index,n,X)

        # cv = score.std() / score.mean()
        # cv2 = score_2.std() / score_2.mean()
        # print(f"Coefficient of variation1: {cv:.3f}")
        # print(f"Coefficient of variation2: {cv2:.3f}")



        # Find worst bottleneck — skip if neighbourhood saturated
        sorted_bottlenecks = torch.argsort(score, descending=True)

        found = False
        for bottleneck_idx in sorted_bottlenecks:
            node_i = edge_index[0, bottleneck_idx].item()
            node_j = edge_index[1, bottleneck_idx].item()

            neigh_i = neigh_dict[node_i]
            neigh_j = neigh_dict[node_j]

            valid_candidates = [
                (u, v) for (u, v) in product(neigh_i, neigh_j)
                if (u, v) not in edge_set
                and (v, u) not in edge_set
                and u != v
            ]

            if not valid_candidates:
                continue

            local_neigh = list(set(neigh_i + neigh_j + [node_i, node_j]))
            old_score = evaluate_neighbourhood(X,edge_index,n,local_neigh,neigh_dict)

            best = (-float('inf'), (-1, -1))
            for (u, v) in valid_candidates:
                rewired_edge_index = add_edge(edge_index, u, v)
                rewired_edge_index = add_edge(rewired_edge_index,v,u)
                rewired_neigh_dict = add_edge_to_neighbour_dict(neigh_dict,u,v)
                new_score = evaluate_neighbourhood(
                    X, rewired_edge_index,n, local_neigh,neigh_dict
                )
                relief = old_score - new_score
                if best[0] < relief:
                    best = (relief, (u, v))

            if best[1] != (-1, -1):
                print("edge_added")
                u, v = best[1]
                edge_index = add_edge(edge_index, u, v)
                edge_index = add_edge(edge_index, v, u)
                edge_set.add((u, v))
                edge_set.add((v, u))
                neigh_dict[u].append(v)
                neigh_dict[v].append(u)

                # Approximate update to LX_smooth for next iteration
                # Avoids full recomputation between exact checkpoints
                deg_u =len(neigh_dict[u])
                deg_v = len(neigh_dict[v])
                diff = LX_smooth[u] - LX_smooth[v]
                LX_smooth[u] = LX_smooth[u] - (1.0/(deg_u+1)) * diff
                LX_smooth[v] = LX_smooth[v] + (1.0/(deg_v+1)) * diff

                found = True
                break

        if not found:
            print(f"Early stop at iteration {idx}: no valid candidates")
            break

    edge_index, _ = add_remaining_self_loops(edge_index)
    return edge_index

# Experiments I : Synthaetic Dataset

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch_geometric.utils import from_networkx, remove_self_loops, get_laplacian
import numpy as np
import networkx as nx
import random
import torch_sparse
from torch import nn

@jit(nopython=True)
def compute_degrees(edge_index, num_nodes=None):
	# returns array of degrees of all nodes
	if num_nodes is None:
		num_nodes = np.max(edge_index) + 1
	degrees = np.zeros(num_nodes)
	m = edge_index.shape[1]
	for i in range(m):
		degrees[edge_index[0, i]] += 1
	return degrees

def build_chain_of_cliques(num_cliques=5, clique_size=20, num_bridge_edges_list=None):
    if num_bridge_edges_list is None:
        num_bridge_edges_list = [1] * (num_cliques - 1)

    assert len(num_bridge_edges_list) == num_cliques - 1

    strength_index = {}
    G = nx.Graph()
    node_offset = 0
    clique_nodes = []
    bridge_edges = []

    # create cliques
    for i in range(num_cliques):
        nodes = list(range(node_offset, node_offset + clique_size))
        G.add_nodes_from(nodes)

        for u in nodes:
            for v in nodes:
                if u != v:
                    G.add_edge(u, v)

        clique_nodes.append(nodes)
        node_offset += clique_size

    # connect cliques with variable edges
    for i in range(num_cliques - 1):
        left_nodes = clique_nodes[i]
        right_nodes = clique_nodes[i + 1]
        num_edges = num_bridge_edges_list[i]

        for _ in range(num_edges):
            u = random.choice(left_nodes)
            v = random.choice(right_nodes)
            G.add_edge(u, v)
            bridge_edges.append((u, v))
            if num_edges not in strength_index:
              strength_index[num_edges] = []

            strength_index[num_edges].append((u,v))


    data = from_networkx(G)
    edge_index = data.edge_index
    n = G.number_of_nodes()

    return edge_index, n, clique_nodes, bridge_edges, strength_index

def dirichlet_energy_grad(edge_index, n, X, edge_weight=None, norm_type=None):
  edge_index, L = get_laplacian(edge_index, edge_weight, norm_type)
  LX = torch_sparse.spmm(edge_index, L, n, n, X)
  return LX

def bottleneck_score(edge_index, n, X, eta=0.1, eps=1e-8):
    LX = dirichlet_energy_grad(edge_index, n, X)
    X_smooth = X - eta * LX

    LX_smooth = dirichlet_energy_grad(edge_index, n, X_smooth)
    diff = torch.norm(
        LX_smooth[edge_index[0]] - LX_smooth[edge_index[1]], dim=-1
    )
    mag = torch.norm(LX_smooth[edge_index[0]] + LX_smooth[edge_index[1]], dim=-1)
    return diff/(mag+eps)

# def bottleneck_score_v2(edge_index, n, X, eta=0.1, eps=1e-8):
#     """
#     Normalised gradient difference — matches tango_like_bottleneck
#     structure but uses Dirichlet energy gradient instead of learned energy.
#     """
#     LX = dirichlet_energy_grad(edge_index, n, X)
#     X_smooth = X - eta * LX

#     i = edge_index[0]
#     j = edge_index[1]


#     diff = torch.norm(X_smooth[i] - X_smooth[j], dim=-1)
#     mag  = torch.norm(X_smooth[i], dim=-1) + torch.norm(X_smooth[j], dim=-1)

#     score = diff/(mag+eps)
#     return score

def bottleneck_score_v2(edge_index, n, X, eta=None, eps=1e-8):

    # Compute max degree to set stable eta
    deg = torch.zeros(n)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.shape[1]))
    d_max = deg.max().item()

    # Stable eta must satisfy eta < 1/d_max
    # Use 0.9/d_max to stay safely below the threshold
    if eta is None:
        eta = 0.9 / (d_max + 1e-8)


    LX = dirichlet_energy_grad(edge_index, n, X)
    X_smooth = X - eta * LX

    i = edge_index[0]
    j = edge_index[1]

    diff = torch.norm(X_smooth[i] - X_smooth[j], dim=-1)
    mag  = torch.norm(X_smooth[i], dim=-1) + torch.norm(X_smooth[j], dim=-1)

    score = diff / (mag + eps)
    return score
def fosr_edge_scores(edge_index, n, num_power_iter=5, initial_power_iters=10):
    edge_index_clean, _ = remove_self_loops(edge_index)
    edge_index_np = edge_index_clean.numpy().astype(np.int64)

    degrees = compute_degrees(edge_index_np, num_nodes=n)

    x = 2 * np.random.random(n) - 1
    for _ in range(initial_power_iters):
        x = x - x.dot(degrees**0.5) * (degrees**0.5) / sum(degrees)
        y = x + adj_matrix_multiply(edge_index_np, x/(degrees**0.5)) / (degrees**0.5)
        x = y / np.linalg.norm(y)

    y = x / ((degrees + 1)**0.5)
    y_torch = torch.from_numpy(y).float()

    # Score existing edges only — no masking needed
    scores = y_torch[edge_index_clean[0]] * y_torch[edge_index_clean[1]]

    return scores


def to_undirected_edge(edge_index, idx):
    u = edge_index[0, idx].item()
    v = edge_index[1, idx].item()
    return tuple(sorted((u, v)))

def snr_analysis(
    num_cliques=5,
    clique_size=25,
    num_bridge_edges_list=[1, 2, 3, 5],
    dim=8,
    signal_strengths=None,
    noise_strengths=None,
    top_k=10,
    num_trials=5, method = 'grad', features = 'uninformative'):
    """
    For each combination of (sigma_signal, sigma_noise), run the bottleneck
    detection and record:
      - mean bridge score
      - mean non-bridge score
      - precision@k (how many bridge edges appear in top-k)
      - separation ratio (bridge_mean / nonbridge_mean)
    """
    if signal_strengths is None:
        signal_strengths = [0.1, 0.25, 0.5, 1.0, 2.0, 4.0]
    if noise_strengths is None:
        noise_strengths = [0.1, 0.25, 0.5, 1.0, 2.0, 4.0]

    # Store results: results[sigma_s][sigma_n] = dict of metrics
    results = {}

    for sigma_s in signal_strengths:
        results[sigma_s] = {}
        for sigma_n in noise_strengths:

            trial_bridge_means = []
            trial_nonbridge_means = []
            trial_precisions = []

            for trial in range(num_trials):
                # Build graph
                edge_index, n, clique_nodes, bridge_edges, strength_index = \
                    build_chain_of_cliques(num_cliques, clique_size, num_bridge_edges_list)

                # Build features with this SNR
                if "informative":
                  X = create_features_chain_snr(n, clique_nodes, dim,sigma_s,sigma_n)
                if "uninformative":
                  X = create_features_uninformative_snr(n, clique_nodes, dim,sigma_s,sigma_n)

                # Compute score
                if method == 'grad':
                  score = bottleneck_score_v2(edge_index, n, X)
                else:
                  score = fosr_edge_scores(edge_index,n)
                # Separate bridge and non-bridge scores
                bridge_undirected = set(tuple(sorted(e)) for e in bridge_edges)
                bridge_scores = []
                non_bridge_scores = []
                seen = set()

                for i in range(score.shape[0]):
                    edge = to_undirected_edge(edge_index, i)
                    if edge in seen:
                        continue
                    seen.add(edge)
                    if edge in bridge_undirected:
                        bridge_scores.append(score[i].item())
                    else:
                        non_bridge_scores.append(score[i].item())

                bridge_mean = np.mean(bridge_scores)
                nonbridge_mean = np.mean(non_bridge_scores)

                # Precision@k
                if method == "grad":
                  sorted_high = torch.argsort(score, descending=True)
                else:
                  sorted_high = torch.argsort(score, descending=False)
                top_edges = set()
                seen_top = set()
                for i in sorted_high:
                    edge = to_undirected_edge(edge_index, i)
                    if edge in seen_top:
                        continue
                    seen_top.add(edge)
                    top_edges.add(edge)
                    if len(top_edges) == top_k:
                        break

                hits = len(top_edges.intersection(bridge_undirected))
                precision = hits / len(bridge_undirected)

                trial_bridge_means.append(bridge_mean)
                trial_nonbridge_means.append(nonbridge_mean)
                trial_precisions.append(precision)

            if method == 'grad':
              seperation = np.mean(trial_bridge_means) / (np.mean(trial_nonbridge_means) + 1e-8)
            else:
              bridge_m = np.mean(trial_bridge_means)
              nonbridge_m = np.mean(trial_nonbridge_means)
              all_means = trial_bridge_means + trial_nonbridge_means
              std = np.std(all_means) + 1e-8
              seperation = (nonbridge_m - bridge_m) / std


            results[sigma_s][sigma_n] = {
                'bridge_mean': np.mean(trial_bridge_means),
                'nonbridge_mean': np.mean(trial_nonbridge_means),
                'precision': np.mean(trial_precisions),
                # separation ratio: how much higher are bridge scores vs non-bridge
                # > 1 means bridge scores higher (correct detection)
                # ~ 1 means no separation (criterion fails)
                'separation': seperation
            }

    return results, signal_strengths, noise_strengths

def create_features_chain_snr(n, clique_nodes, dim, sigma_signal, sigma_noise):
    """Parameterised version of create_features_chain with explicit SNR control."""
    X = torch.zeros((n, dim))
    for nodes in clique_nodes:
        base = torch.randn(dim) * sigma_signal
        noise = torch.randn(len(nodes), dim) * sigma_noise
        X[nodes] = base + noise
    return X

def create_features_uninformative_snr(n, clique_nodes, dim=8, sigma_signal=0.5,
                                       sigma_noise=0.5):
    """
    All nodes get the same base vector — no inter-clique signal.
    Topology has a bottleneck but features give no reason to cross it.

    sigma_noise controls intra-clique variation.
    There is no sigma_signal parameter because by definition
    the inter-clique signal is zero — all cliques share the same base.
    """
    X = torch.zeros((n, dim))

    shared_base = torch.randn(dim) * sigma_signal

    for nodes in clique_nodes:
        noise = torch.randn(len(nodes), dim) * sigma_noise
        X[nodes] = shared_base + noise

    return X


def plot_snr_results(results, signal_strengths, noise_strengths,method):

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # --- Plot 1: Separation ratio heatmap ---
    # separation > 1 means bridge edges score higher than non-bridge (correct)
    # separation ~ 1 means criterion cannot distinguish
    sep_matrix = np.array([
        [results[ss][sn]['separation'] for sn in noise_strengths]
        for ss in signal_strengths
    ])

    if method == 'grad':
      vmin_sep, vmax_sep = 0.5, 3.0
      sep_title = 'Separation Ratio (bridge/non-bridge)\n>1 = correct (green)'
    else:
        sep_vals = sep_matrix.flatten()
        vmin_sep = sep_vals.min()
        vmax_sep = sep_vals.max()
        sep_title = 'Separation Difference (nonbridge - bridge)\n>0 = correct (green)'


    im = axes[0].imshow(sep_matrix, aspect='auto', cmap='RdYlGn',
                        vmin=0.5, vmax=3.0)
    axes[0].set_xticks(range(len(noise_strengths)))
    axes[0].set_yticks(range(len(signal_strengths)))
    axes[0].set_xticklabels([f'{s:.2f}' for s in noise_strengths])
    axes[0].set_yticklabels([f'{s:.2f}' for s in signal_strengths])
    axes[0].set_xlabel('sigma_noise')
    axes[0].set_ylabel('sigma_signal')
    axes[0].set_title('Separation Ratio (bridge / non-bridge)\n>1 = correct detection (green)')
    plt.colorbar(im, ax=axes[0])

    # Draw the SNR=1 diagonal (sigma_signal = sigma_noise)
    # Points above diagonal: signal > noise (should work)
    # Points below diagonal: noise > signal (may fail)
    for i in range(len(signal_strengths)):
        for j in range(len(noise_strengths)):
            axes[0].text(j, i, f'{sep_matrix[i,j]:.2f}',
                        ha='center', va='center', fontsize=8)

    # --- Plot 2: Precision@k heatmap ---
    prec_matrix = np.array([
        [results[ss][sn]['precision'] for sn in noise_strengths]
        for ss in signal_strengths
    ])

    im2 = axes[1].imshow(prec_matrix, aspect='auto', cmap='RdYlGn',
                         vmin=0, vmax=1.0)
    axes[1].set_xticks(range(len(noise_strengths)))
    axes[1].set_yticks(range(len(signal_strengths)))
    axes[1].set_xticklabels([f'{s:.2f}' for s in noise_strengths])
    axes[1].set_yticklabels([f'{s:.2f}' for s in signal_strengths])
    axes[1].set_xlabel('sigma_noise')
    axes[1].set_ylabel('sigma_signal')
    axes[1].set_title('Precision@k\n(fraction of bridge edges in top-k)')
    plt.colorbar(im2, ax=axes[1])

    for i in range(len(signal_strengths)):
        for j in range(len(noise_strengths)):
            axes[1].text(j, i, f'{prec_matrix[i,j]:.2f}',
                        ha='center', va='center', fontsize=8)

    # --- Plot 3: Line plot — precision vs SNR ratio for fixed noise levels ---
    # SNR ratio = sigma_signal / sigma_noise
    # This is the clearest single summary plot
    for sigma_n in noise_strengths:
        snr_ratios = [ss / sigma_n for ss in signal_strengths]
        precisions = [results[ss][sigma_n]['precision'] for ss in signal_strengths]
        axes[2].plot(snr_ratios, precisions, marker='o',
                    label=f'noise={sigma_n:.2f}')

    axes[2].axvline(x=1.0, color='black', linestyle='--', alpha=0.5,
                   label='SNR=1 (signal=noise)')
    axes[2].set_xlabel('SNR ratio (sigma_signal / sigma_noise)')
    axes[2].set_ylabel('Precision@k')
    axes[2].set_title('Detection precision vs SNR\nVertical line: SNR=1')
    axes[2].legend(fontsize=8)
    axes[2].set_ylim(0, 1.1)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('snr_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    return fig


# Run the analysis
results, signal_strengths, noise_strengths = snr_analysis(
    num_cliques=2,
    clique_size=25,
    num_bridge_edges_list=[1],
    dim=8,
    signal_strengths=[0.1, 0.25, 0.5, 1.0, 2.0, 4.0],
    noise_strengths=[0.1, 0.25, 0.5, 1.0, 2.0, 4.0],
    top_k=2,
    num_trials=11,
    method = 'fosr',features = 'uninformative'
)

plot_snr_results(results, signal_strengths, noise_strengths,method='fosr')

In [ ]:
import torch
import networkx as nx
from torch_geometric.utils import from_networkx, remove_self_loops, get_laplacian
import numpy as np


def create_informative_gradient_graph(n=100, p=0.3, dim=8,
                                       sigma_signal=2.0, sigma_noise=0.3):
    """
    Dense random graph — no topology bottleneck.
    Features vary systematically across a latent boundary.

    sigma_signal: magnitude of inter-group feature difference
    sigma_noise: magnitude of per-node noise within each group
    SNR = sigma_signal / sigma_noise
    """
    G = nx.erdos_renyi_graph(n, p)
    while not nx.is_connected(G):
        G = nx.erdos_renyi_graph(n, p)

    data = from_networkx(G)
    edge_index = data.edge_index

    half = n // 2
    X = torch.zeros((n, dim))

    # Group A and B have different base vectors
    base_A = torch.randn(dim) * sigma_signal
    base_B = torch.randn(dim) * sigma_signal

    # Per-node noise within each group
    noise_A = torch.randn(half, dim) * sigma_noise
    noise_B = torch.randn(n - half, dim) * sigma_noise

    X[:half] = base_A + noise_A
    X[half:] = base_B + noise_B

    return edge_index, n, X, half


def get_cross_boundary_edges(edge_index, half):
    """
    Returns set of undirected edges that cross the group boundary.
    These are the task-relevant edges your criterion should find.
    """
    cross = set()
    for i in range(edge_index.shape[1]):
        u = edge_index[0, i].item()
        v = edge_index[1, i].item()
        # Cross-boundary: one node in each group
        if (u < half and v >= half) or (u >= half and v < half):
            cross.add(tuple(sorted((u, v))))
    return cross

def fosr_edge_scores(edge_index, n, num_power_iter=1, initial_power_iters=10):
    edge_index_clean, _ = remove_self_loops(edge_index)
    edge_index_np = edge_index_clean.numpy().astype(np.int64)

    degrees = compute_degrees(edge_index_np, num_nodes=n)

    x = 2 * np.random.random(n) - 1
    for _ in range(initial_power_iters):
        x = x - x.dot(degrees**0.5) * (degrees**0.5) / sum(degrees)
        y = x + adj_matrix_multiply(edge_index_np, x/(degrees**0.5)) / (degrees**0.5)
        x = y / np.linalg.norm(y)

    y = x / ((degrees + 1)**0.5)
    y_torch = torch.from_numpy(y).float()

    # Score existing edges only — no masking needed
    scores = y_torch[edge_index_clean[0]] * y_torch[edge_index_clean[1]]

    return scores
def bottleneck_score_v2(edge_index, n, X, eta=None, eps=1e-8):

    # Compute max degree to set stable eta
    deg = torch.zeros(n)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.shape[1]))
    d_max = deg.max().item()

    # Stable eta must satisfy eta < 1/d_max
    # Use 0.9/d_max to stay safely below the threshold
    if eta is None:
        eta = 0.9 / (d_max + 1e-8)


    LX = dirichlet_energy_grad(edge_index, n, X)
    X_smooth = X - eta * LX

    i = edge_index[0]
    j = edge_index[1]

    diff = torch.norm(X_smooth[i] - X_smooth[j], dim=-1)
    mag  = torch.norm(X_smooth[i], dim=-1) + torch.norm(X_smooth[j], dim=-1)

    score = diff / (mag + eps)
    return score

def bottleneck_score(edge_index, n, X, eta=0.1, eps=1e-8):
    LX = dirichlet_energy_grad(edge_index, n, X)
    X_smooth = X - eta * LX

    LX_smooth = dirichlet_energy_grad(edge_index, n, X_smooth)
    diff = torch.norm(
        LX_smooth[edge_index[0]] - LX_smooth[edge_index[1]], dim=-1
    )
    mag = torch.norm(LX_smooth[edge_index[0]] + LX_smooth[edge_index[1]], dim=-1)
    return diff/(mag+eps)

def run_graph_b_snr_analysis(
    n=100, p=0.3, dim=8,
    signal_strengths=None,
    noise_strengths=None,
    top_k=20,
    num_trials=5, method = 'grad'
):
    """
    For Graph B: dense graph with informative feature gradient.
    Metric: cross-boundary precision@k — what fraction of top-k
    scored edges actually cross the feature boundary.
        """
    if signal_strengths is None:
        signal_strengths = [0.1, 0.25, 0.5, 1.0, 2.0, 4.0]
    if noise_strengths is None:
        noise_strengths = [0.1, 0.25, 0.5, 1.0, 2.0, 4.0]

    results = {}

    for sigma_s in signal_strengths:
        results[sigma_s] = {}
        for sigma_n in noise_strengths:

            trial_precisions = []
            trial_separations = []
            trial_random_baselines = []

            for trial in range(num_trials):
                edge_index, n_nodes, X, half = create_informative_gradient_graph(
                    n=n, p=p, dim=dim,
                    sigma_signal=sigma_s,
                    sigma_noise=sigma_n
                )

                # Compute your criterion score

                if method == "grad":
                  score = bottleneck_score(edge_index,n_nodes,X)
                else:
                  score = fosr_edge_scores(edge_index,n)

                # Identify cross-boundary edges
                cross_boundary = get_cross_boundary_edges(edge_index, half)
                total_cross = len(cross_boundary)

                # Compute random baseline precision
                # In a random graph with p=0.3, fraction of cross-boundary edges
                # is approximately (half * (n-half)) / total_edges * 2
                total_undirected = edge_index.shape[1] // 2
                random_baseline = total_cross / total_undirected

                # Get top-k edges by score
                if method == "grad":
                  sorted_high = torch.argsort(score, descending=True)
                else:
                  sorted_high = torch.argsort(score, descending=False)
                top_edges = set()
                seen = set()
                for i in sorted_high:
                    u = edge_index[0, i].item()
                    v = edge_index[1, i].item()
                    key = tuple(sorted((u, v)))
                    if key in seen:
                        continue
                    seen.add(key)
                    top_edges.add(key)
                    if len(top_edges) == top_k:
                        break

                # Cross-boundary precision: fraction of top-k that cross boundary
                hits = len(top_edges.intersection(cross_boundary))
                precision = hits / top_k

                # Separation: mean score of cross-boundary vs within-group edges
                cross_scores = []
                within_scores = []
                seen2 = set()
                for i in range(score.shape[0]):
                    u = edge_index[0, i].item()
                    v = edge_index[1, i].item()
                    key = tuple(sorted((u, v)))
                    if key in seen2:
                        continue
                    seen2.add(key)
                    if key in cross_boundary:
                        cross_scores.append(score[i].item())
                    else:
                        within_scores.append(score[i].item())

                cross_m = np.mean(cross_scores)
                within_m = np.mean(within_scores)

                if method == 'grad':
                    separation = cross_m / (within_m + 1e-8)
                else:
                    all_scores = cross_scores + within_scores
                    std = np.std(all_scores) + 1e-8
                    separation = (within_m - cross_m) / std

                trial_precisions.append(precision)
                trial_separations.append(separation)
                trial_random_baselines.append(random_baseline)


            results[sigma_s][sigma_n] = {
                'precision_mean': np.mean(trial_precisions),
                'precision_std': np.std(trial_precisions),
                'separation': np.mean(trial_separations),
                # How much better than random?
                # lift > 1 means your method finds cross-boundary edges
                # above chance — even without topology bottleneck
                'lift': np.mean(trial_precisions) / (np.mean(trial_random_baselines) + 1e-8),
                'random_baseline': np.mean(trial_random_baselines)
            }

    return results, signal_strengths, noise_strengths


def plot_graph_b_results(results, signal_strengths, noise_strengths,method ='fosr'):
    """
    Three plots for Graph B:
    1. Separation ratio heatmap (cross-boundary vs within-group scores)
    2. Lift over random baseline heatmap
       (lift=1 means no better than random, lift>1 means your method works)
    3. Line plot of lift vs SNR ratio
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # --- Plot 1: Separation ratio ---
    sep_matrix = np.array([
        [results[ss][sn]['separation'] for sn in noise_strengths]
        for ss in signal_strengths
    ])

    if method == 'grad':
        vmin_sep, vmax_sep = 0.5, 3.0
        sep_title = 'Separation Ratio (bridge/non-bridge)\n>1 = correct (green)'
    else:
        sep_vals = sep_matrix.flatten()
        vmin_sep = sep_vals.min()
        vmax_sep = max(sep_vals.max(), 1e-8)  # avoid vmin==vmax
        sep_title = "Separation Cohen's d (nonbridge-bridge)/std\n>0 = correct (green)"



    im1 = axes[0].imshow(sep_matrix, aspect='auto', cmap='RdYlGn',
                         vmin=0.5, vmax=3.0)
    axes[0].set_xticks(range(len(noise_strengths)))
    axes[0].set_yticks(range(len(signal_strengths)))
    axes[0].set_xticklabels([f'{s:.2f}' for s in noise_strengths])
    axes[0].set_yticklabels([f'{s:.2f}' for s in signal_strengths])
    axes[0].set_xlabel('sigma_noise')
    axes[0].set_ylabel('sigma_signal')
    axes[0].set_title('Separation Ratio\n(cross-boundary / within-group scores)')
    plt.colorbar(im1, ax=axes[0])
    for i in range(len(signal_strengths)):
        for j in range(len(noise_strengths)):
            axes[0].text(j, i, f'{sep_matrix[i,j]:.2f}',
                        ha='center', va='center', fontsize=8)

    # --- Plot 2: Lift over random baseline ---
    # This is the key plot for Graph B
    # Lift = precision@k / random_baseline_precision
    # If SDRF/FoSR select edges randomly on this graph, their lift = 1
    # Your method should show lift > 1 in high-SNR regime
    lift_matrix = np.array([
        [results[ss][sn]['lift'] for sn in noise_strengths]
        for ss in signal_strengths
    ])

    im2 = axes[1].imshow(lift_matrix, aspect='auto', cmap='RdYlGn',
                         vmin=0.5, vmax=5.0)
    axes[1].set_xticks(range(len(noise_strengths)))
    axes[1].set_yticks(range(len(signal_strengths)))
    axes[1].set_xticklabels([f'{s:.2f}' for s in noise_strengths])
    axes[1].set_yticklabels([f'{s:.2f}' for s in signal_strengths])
    axes[1].set_xlabel('sigma_noise')
    axes[1].set_ylabel('sigma_signal')
    axes[1].set_title('Lift over Random Baseline\n=(1.0)')
    plt.colorbar(im2, ax=axes[1])
    for i in range(len(signal_strengths)):
        for j in range(len(noise_strengths)):
            axes[1].text(j, i, f'{lift_matrix[i,j]:.2f}',
                        ha='center', va='center', fontsize=8)

    # Draw lift=1 reference line in text
    axes[1].set_title(
        'Lift over Random Baseline\n'
        f'Random baseline ≈ {results[signal_strengths[0]][noise_strengths[0]]["random_baseline"]:.2f}'
    )

    # --- Plot 3: Lift vs SNR ratio ---
    for sigma_n in noise_strengths:
        snr_ratios = [ss / sigma_n for ss in signal_strengths]
        lifts = [results[ss][sigma_n]['lift'] for ss in signal_strengths]
        axes[2].plot(snr_ratios, lifts, marker='o',
                    label=f'noise={sigma_n:.2f}')

    axes[2].axhline(y=1.0, color='black', linestyle='--', alpha=0.7,
                   label='Random baseline')
    axes[2].axvline(x=1.0, color='gray', linestyle=':', alpha=0.5,
                   label='SNR=1')
    axes[2].set_xlabel('SNR ratio (sigma_signal / sigma_noise)')
    axes[2].set_ylabel('Lift over random')
    axes[2].set_title(
        'Graph B: Cross-boundary detection lift vs SNR\n'
    )
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)
    axes[2].set_ylim(0, None)

    plt.suptitle(
        'Graph B: Dense graph, no topology bottleneck, informative feature gradient\n',
        fontsize=11, y=1.02
    )
    plt.tight_layout()
    plt.savefig('graph_b_snr_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    return fig


# Run it
results_b, ss, sn = run_graph_b_snr_analysis(
    n=100, p=0.3, dim=8,
    signal_strengths=[0.1, 0.25, 0.5, 1.0, 2.0, 4.0],
    noise_strengths=[0.1, 0.25, 0.5, 1.0, 2.0, 4.0],
    top_k=20,
    num_trials=5, method = 'fosr'
)

plot_graph_b_results(results_b, ss, sn)

In [4]:
def compute_feature_snr(data):
    X = data.x.numpy()
    y = data.y.numpy()
    classes = np.unique(y)

    print(f"Feature matrix shape: {X.shape}")
    print(f"Feature value range: [{X.min():.4f}, {X.max():.4f}]")
    print(f"Feature mean norm: {np.linalg.norm(X, axis=1).mean():.4f}")

    # Global mean
    global_mean = X.mean(axis=0)  # (d,)

    # Total variance — variance of all nodes around global mean
    total_var = ((X - global_mean)**2).sum(axis=1).mean()

    # Inter-class variance — variance of class means around global mean
    # weighted by class size
    inter_var = 0.0
    intra_vars = []
    community_means = []

    for c in classes:
        mask = y == c
        n_c = mask.sum()
        if n_c < 2:
            continue
        class_features = X[mask]           # (n_c, d)
        class_mean = class_features.mean(axis=0)  # (d,)
        community_means.append(class_mean)

        # Inter: squared distance of class mean from global mean
        # weighted by class size — between-class scatter
        inter_var += n_c * ((class_mean - global_mean)**2).sum()

        # Intra: mean squared distance of nodes from their class mean
        intra_var_c = ((class_features - class_mean)**2).sum(axis=1).mean()
        intra_vars.append(intra_var_c)

    inter_var /= len(X)  # normalise by total n
    intra_var_mean = np.mean(intra_vars)

    # Explained variance ratio — standard metric from LDA
    # High = class labels explain a large fraction of feature variance
    explained_variance_ratio = inter_var / (total_var + 1e-8)
    snr = inter_var / (intra_var_mean + 1e-8)

    print(f"\nNumber of classes: {len(classes)}")
    print(f"Total feature variance:            {total_var:.6f}")
    print(f"Inter-class feature variance:      {inter_var:.6f}")
    print(f"Mean intra-class feature variance: {intra_var_mean:.6f}")
    print(f"Feature SNR (inter/intra):         {snr:.4f}")
    print(f"Explained variance ratio:          {explained_variance_ratio:.4f}")
    print(f"(Explained variance > 0.1 suggests class-informative features)")

    return snr, explained_variance_ratio